# 365 Probabilidades - Dia #111
## Qual a probabilidade de o seu dia acontecer sem você?

**Tipo:** Comportamental
**Data de publicação:** 2026-10-02
**Ferramenta:** Python
**Decisão analisada:** Quanto do meu dia eu realmente decido?
**Hashtag:** #365Probabilidades #Dia111

---

### 📖 A História

Existe um número que circula há vinte anos em livro de hábito, palestra de
produtividade e legenda de rede social: **43% do seu dia é automático.**

É um número bom. Tem artigo por trás, tem revista séria, tem amostra grande. E, ao
contrário da maioria dos números que essa série já desmontou, ele não está errado.

O mesmo artigo publicou também 35%, no estudo anterior, com protocolo parecido mas não
igual. E publicou uma terceira coisa, que ninguém repete: o desvio-padrão entre as
pessoas. Dezenove pontos, numa escala que só tem cem.

Hoje o modelo não serve para derrubar o número. Serve para mostrar que ele responde bem
a uma pergunta ("qual é a média desse grupo?") e mal a outra ("quanto da minha vida é
hábito?").

---

### 📚 O Conceito: hábito é comportamento que dispensa decisão

Wendy Wood, Jeffrey Quinn e Deborah Kashy definiram hábito de um jeito operacional:
comportamento repetido **quase todo dia** e **geralmente no mesmo lugar**. Os dois
critérios juntos, não um ou outro.

A ideia por trás é que a repetição em contexto estável transfere o controle da ação da
intenção para a deixa ambiental. Você não decide escovar os dentes. O banheiro decide
por você.

Para medir isso fora do laboratório, os autores usaram amostragem por sinal: um relógio
apitava de hora em hora e a pessoa anotava o que estava fazendo, pensando e sentindo
naquele instante. Depois classificava cada comportamento pelos dois critérios.

O Estudo 1 acompanhou 70 estudantes por um dia e achou 35% dos comportamentos
habituais. O Estudo 2 acompanhou 209 estudantes por dois dias e achou 43%.

O 43% é o que sobreviveu na cultura.

---

### 🧮 O Modelo

Quatro camadas. A primeira testa a precisão da média, a segunda testa se os dois
números são diferentes, e as duas últimas perguntam de quem é o dia.

1. **A média, contada de dois jeitos.** Posterior de Jeffreys contando cada anotação
   como observação (a leitura ingênua, que produz o intervalo estreito que todo mundo
   cita) contra o intervalo pela unidade certa, a pessoa, usando a média e o
   desvio-padrão entre participantes que o artigo publica.
2. **35% e 43% são diferentes?** Teste da diferença entre as médias dos dois estudos,
   contando pessoas, e o que os próprios autores dizem sobre a origem da diferença.
   A correlação intraclasse é derivada do DP publicado, não assumida.
3. **Heterogeneidade entre pessoas.** Beta calibrada pelo método dos momentos na média
   43% e no DP 19 publicados, para descrever não a média dos dias, mas a distribuição
   deles. Versão de robustez descontando o ruído amostral dentro de cada pessoa.
4. **Simulação de sequências.** Dez momentos observados por pessoa, semente declarada,
   comparando o mundo real com um mundo em que todo mundo fosse a média. Momentos, não
   horas: o estudo classificou comportamentos, não tempo.

**Fontes:**
- Wood, W., Quinn, J. M. & Kashy, D. A. (2002). "Habits in Everyday Life: Thought,
  Emotion, and Action". *Journal of Personality and Social Psychology*, 83(6),
  1281-1297. Estudo 1: N=70, um dia, média de 9,58 anotações por pessoa, proporção
  habitual **M = 0,35, DP = 0,19** (Tabela 1). Estudo 2: N=209, dois dias, 20,74
  anotações por pessoa, **M = 0,43, DP = 0,19** (Tabela 2). No Estudo 1 a pessoa
  registrava um comportamento por sinal; no Estudo 2 podia registrar vários. Os autores
  atribuem o 43% maior a essa mudança de protocolo.

**Nota metodológica sobre o fator ×0,80.** Não aplicado, e o motivo importa. O fator
existe para descontar o exagero de quem se autodeclara em survey populacional, quando o
autorrelato é o ruído entre a pergunta e o mundo. Aqui o autorrelato **é** a medida:
"quase todo dia, no mesmo lugar" é a definição da variável, não uma aproximação dela.
Cortar 20% do 43% seria cortar 20% da definição. Segue a decisão tomada no **#063**.
A incerteza entra por onde ela de fato existe, que é a variação entre pessoas, e isso
alarga a distribuição em vez de empurrar o ponto.

**Correlação intraclasse:** o artigo não a publica como tal, mas o DP entre pessoas
permite derivá-la (camada 2). Nenhuma camada depende de valor assumido.

**Ponte interna:** o **#006** mostrou disciplina prevendo resultado melhor que
inteligência. Lá a repetição era virtude. Aqui ela é anestesia. É o mesmo mecanismo
visto dos dois lados.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
SORTEIOS = 500_000

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'

def mil(n):
    return f"{int(n):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA ---
# Wood, Quinn & Kashy (2002), JPSP 83(6), 1281-1297.
# Amostragem por sinal: relógio apita de hora em hora, a pessoa anota o que está fazendo.
# p e sd: média e desvio-padrão ENTRE PARTICIPANTES da proporção de comportamentos
# habituais (Tabela 1 e Tabela 2 do artigo).

ESTUDOS = {
    'Estudo 1': dict(p=0.35, sd=0.19, pessoas=70,  por_pessoa=9.58,  dias=1),
    'Estudo 2': dict(p=0.43, sd=0.19, pessoas=209, por_pessoa=20.74, dias=2),
}
for e in ESTUDOS.values():
    e['registros'] = round(e['pessoas'] * e['por_pessoa'])

MOMENTOS = 10   # observações de comportamento por sequência simulada (camada 4).
                # Momentos, não horas: o estudo classificou comportamentos, não tempo.

print("=" * 70)
print("  DADOS DA LITERATURA · WOOD, QUINN & KASHY (2002)")
print("=" * 70)
print("\n  Critério de hábito: 'quase todo dia' E 'geralmente no mesmo lugar'.")
print("  Os dois juntos. Classificação feita pelo próprio participante.\n")
for nome, e in ESTUDOS.items():
    print(f"  {nome}: {e['pessoas']:>3} pessoas · {e['dias']} dia"
          f"{'s' if e['dias'] > 1 else ' '} · "
          f"{vir(e['por_pessoa'], 2)} anotações por pessoa · "
          f"{mil(e['registros'])} registros")
    print(f"  {'':9} proporção habitual por pessoa: média {e['p'] * 100:.0f}%, "
          f"DP {e['sd'] * 100:.0f} pontos")
print(f"\n  O número que a cultura guardou: {ESTUDOS['Estudo 2']['p'] * 100:.0f}%.")
print(f"  O número que ninguém repete:    DP de {ESTUDOS['Estudo 2']['sd'] * 100:.0f} pontos "
      f"entre as pessoas.")
print(f"\n  Protocolo: um comportamento por sinal no Estudo 1, vários no Estudo 2.")
print(f"  Os autores atribuem o 43% maior a essa diferença.")
print(f"  ×0,80: não aplicado. O autorrelato aqui é a medida, não o ruído. Ver #063.")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# ── CAMADA 1 · a média, contada de dois jeitos ──
def jeffreys(p, n):
    x = p * n
    return stats.beta(x + 0.5, n - x + 0.5)

def ic(dist):
    return dist.interval(0.95)

def ic_pessoas(e):
    """IC 95% da média usando a pessoa como unidade: média ± t · DP/√n."""
    t = stats.t.ppf(0.975, e['pessoas'] - 1)
    se = e['sd'] / np.sqrt(e['pessoas'])
    return e['p'] - t * se, e['p'] + t * se, se

print("=" * 70)
print("  CAMADA 1 · A MÉDIA, CONTADA DE DOIS JEITOS")
print("=" * 70)
for nome, e in ESTUDOS.items():
    lo_a, hi_a = ic(jeffreys(e['p'], e['registros']))
    lo_p, hi_p, se = ic_pessoas(e)
    e.update(ic_anot=(lo_a, hi_a), ic_pess=(lo_p, hi_p), se=se)
    print(f"\n  {nome}: {e['p'] * 100:.0f}%")
    print(f"    contando anotação (n = {mil(e['registros'])}): "
          f"[{vir(lo_a * 100)}%; {vir(hi_a * 100)}%] · largura {vir((hi_a - lo_a) * 100)} pp")
    print(f"    contando pessoa   (n = {e['pessoas']:>5}, DP {e['sd'] * 100:.0f}): "
          f"[{vir(lo_p * 100)}%; {vir(hi_p * 100)}%] · largura {vir((hi_p - lo_p) * 100)} pp")
e2 = ESTUDOS['Estudo 2']
print(f"\n  Contando pessoas, o intervalo do 43% abre de "
      f"{vir((e2['ic_anot'][1] - e2['ic_anot'][0]) * 100)} para "
      f"{vir((e2['ic_pess'][1] - e2['ic_pess'][0]) * 100)} pp. Abre, mas pouco.")
print(f"  A média é firme: entre {vir(e2['ic_pess'][0] * 100, 0)}% e "
      f"{vir(e2['ic_pess'][1] * 100, 0)}%.")

# ── CAMADA 2 · 35% e 43% são números diferentes? ──
e1 = ESTUDOS['Estudo 1']
dif = e2['p'] - e1['p']
se_d = np.sqrt(e1['se'] ** 2 + e2['se'] ** 2)
df_w = se_d ** 4 / (e1['se'] ** 4 / (e1['pessoas'] - 1) + e2['se'] ** 4 / (e2['pessoas'] - 1))
t_obs = dif / se_d
p_val = 2 * stats.t.sf(t_obs, df_w)
t_crit = stats.t.ppf(0.975, df_w)
dif_lo, dif_hi = dif - t_crit * se_d, dif + t_crit * se_d

print("\n" + "=" * 70)
print("  CAMADA 2 · 35% E 43% SÃO NÚMEROS DIFERENTES? (contando pessoas)")
print("=" * 70)
print(f"\n  Diferença = {vir(dif * 100)} pp · EP = {vir(se_d * 100, 2)} pp · "
      f"t de Welch = {vir(t_obs, 2)} (gl ≈ {df_w:.0f}) · p = {vir(p_val, 4)}")
print(f"  IC 95% da diferença: [{vir(dif_lo * 100)}; {vir(dif_hi * 100)}] pp · "
      f"zero {'DENTRO' if dif_lo < 0 < dif_hi else 'FORA'} do intervalo")
print(f"\n  A diferença é real mesmo pela unidade certa. E tem explicação publicada:")
print(f"  no Estudo 2 a pessoa podia listar vários comportamentos por sinal, o que")
print(f"  puxa mais hábitos para dentro. Não é sorteio. É protocolo.")

# ── CAMADA 2b · a correlação intraclasse, derivada em vez de assumida ──
var_tot = e2['p'] * (1 - e2['p'])              # variância de Bernoulli com p = 0,43
icc_bruta = e2['sd'] ** 2 / var_tot            # DP publicado, com ruído dentro da pessoa
ruido = var_tot / e2['por_pessoa']             # variância amostral de uma proporção com m obs.
sd_real = np.sqrt(e2['sd'] ** 2 - ruido)       # heterogeneidade verdadeira entre pessoas
icc_real = sd_real ** 2 / var_tot
deff = 1 + (e2['por_pessoa'] - 1) * icc_bruta
obs_efetivas = e2['por_pessoa'] / deff
n_ef = e2['registros'] / deff
lo_ef, hi_ef = ic(jeffreys(e2['p'], n_ef))

print("\n" + "=" * 70)
print("  CAMADA 2b · O QUE CONTA COMO AMOSTRA")
print("=" * 70)
print(f"\n  ICC implícita no DP publicado:      {vir(icc_bruta, 3)}")
print(f"  ICC descontando ruído amostral:     {vir(icc_real, 3)}  "
      f"(DP verdadeiro entre pessoas ≈ {vir(sd_real * 100)} pontos)")
print(f"  Efeito de desenho 1 + (m − 1)·ρ:    {vir(deff, 2)}")
print(f"  → {vir(e2['por_pessoa'], 2)} anotações de uma pessoa valem cerca de "
      f"{vir(obs_efetivas, 1)} observações")
print(f"  → n efetivo ≈ {mil(round(n_ef))} · IC via efeito de desenho "
      f"[{vir(lo_ef * 100)}%; {vir(hi_ef * 100)}%], coerente com a camada 1")

# ── CAMADA 3 · a distribuição das pessoas, não a média delas ──
def beta_momentos(media, dp):
    k = media * (1 - media) / dp ** 2 - 1
    return stats.beta(media * k, (1 - media) * k)

pessoas = beta_momentos(e2['p'], e2['sd'])
alfa, beta_ = pessoas.args
d10, d90 = pessoas.ppf(0.10), pessoas.ppf(0.90)
pessoas_rob = beta_momentos(e2['p'], sd_real)
r10, r90 = pessoas_rob.ppf(0.10), pessoas_rob.ppf(0.90)

print("\n" + "=" * 70)
print("  CAMADA 3 · O SEU DIA NÃO É O DIA MÉDIO")
print("=" * 70)
print(f"\n  Beta pelo método dos momentos, média {e2['p'] * 100:.0f}% e DP {e2['sd'] * 100:.0f} "
      f"(dados do artigo)")
print(f"  Beta(α = {vir(alfa, 2)}; β = {vir(beta_, 2)})\n")
print(f"  Média entre pessoas:                    {vir(pessoas.mean() * 100)}%")
print(f"  Os 80% do meio vão de                   {vir(d10 * 100)}% a {vir(d90 * 100)}%")
print(f"  P(mais da metade dos comportamentos habituais) = {vir(pessoas.sf(0.50) * 100)}%")
print(f"  P(menos de um quinto habitual)                 = {vir(pessoas.cdf(0.20) * 100)}%")
print(f"\n  Robustez (DP descontado do ruído, {vir(sd_real * 100)} pontos): "
      f"Beta({vir(pessoas_rob.args[0], 2)}; {vir(pessoas_rob.args[1], 2)}), "
      f"80% do meio de {vir(r10 * 100)}% a {vir(r90 * 100)}%.")
print(f"  A faixa encolhe um pouco e continua cobrindo quase metade da escala.")

# ── CAMADA 4 · sequências de dez momentos, dois mundos ──
g = np.random.default_rng(SEMENTE)
p_pessoa = g.beta(alfa, beta_, SORTEIOS)
hab = g.binomial(MOMENTOS, p_pessoa)                 # mundo heterogêneo
hab_med = g.binomial(MOMENTOS, e2['p'], SORTEIOS)    # mundo em que todos são a média

print("\n" + "=" * 70)
print(f"  CAMADA 4 · {MOMENTOS} MOMENTOS OBSERVADOS, {mil(SORTEIOS)} SEQUÊNCIAS SIMULADAS")
print("=" * 70)
print(f"\n  {'':38}{'pessoas diferentes':>20}{'todos iguais à média':>24}")
print(f"  {'momentos habituais (média)':<38}{vir(hab.mean(), 2):>20}{vir(hab_med.mean(), 2):>24}")
print(f"  {'os 80% do meio':<38}"
      f"{f'{np.percentile(hab, 10):.0f} a {np.percentile(hab, 90):.0f}':>20}"
      f"{f'{np.percentile(hab_med, 10):.0f} a {np.percentile(hab_med, 90):.0f}':>24}")
print(f"  {'P(sequência sem nenhum habitual)':<38}{vir((hab == 0).mean() * 100) + '%':>20}"
      f"{vir((hab_med == 0).mean() * 100, 2) + '%':>24}")
print(f"  {'P(sequência toda habitual)':<38}{vir((hab == MOMENTOS).mean() * 100) + '%':>20}"
      f"{vir((hab_med == MOMENTOS).mean() * 100, 2) + '%':>24}")
print(f"\n  A média sobrevive: {vir(hab.mean(), 2)} contra {vir(hab_med.mean(), 2)}.")
print(f"  A distribuição não: as pessoas se espalham muito mais do que a média sugere.")
print("=" * 70)

In [ ]:
# --- VISUALIZAÇÃO ---

# ══ GRÁFICO 1 · os dois números do mesmo artigo ══
fig1, ax1 = plt.subplots(figsize=(11, 6.5))

linhas = []
for nome, e in ESTUDOS.items():
    linhas.append((f"{nome}\ncada anotação conta", e['p'], *e['ic_anot'], CINZA))
    linhas.append((f"{nome}\ncada pessoa conta", e['p'], *e['ic_pess'], VERMELHO))

ys = [3.25, 2.75, 1.25, 0.75]
for y, (rot, p, lo, hi, cor) in zip(ys, linhas):
    ax1.plot([lo * 100, hi * 100], [y, y], color=cor, linewidth=6, alpha=0.35,
             solid_capstyle='round')
    ax1.plot(p * 100, y, 'o', color=cor, markersize=11, zorder=3)
    ax1.text(hi * 100 + 0.8, y, f"[{vir(lo * 100)}; {vir(hi * 100)}]",
             va='center', fontsize=10, color=cor)

ax1.set_yticks(ys)
ax1.set_yticklabels([l[0] for l in linhas], fontsize=10)
ax1.set_xlim(26, 52)
ax1.set_ylim(0.2, 3.8)
ax1.set_xlabel('Comportamentos classificados como habituais (%)')
ax1.set_title('O mesmo artigo publicou 35% e 43%\n'
              f'Contando pessoa, a diferença continua: [{vir(dif_lo * 100)}; '
              f'{vir(dif_hi * 100)}] pontos, e os autores explicam por quê',
              fontsize=13, pad=15)
ax1.axvline(43, color=DOURADO, linestyle='--', linewidth=1.5, alpha=0.8)
ax1.text(43.5, 2.05, 'o número que a cultura guardou', fontsize=10, color=DOURADO)
ax1.grid(axis='y', alpha=0)

plt.figtext(0.5, 0.005, 'Fonte: Wood, Quinn & Kashy (2002), JPSP 83(6), Tabelas 1 e 2 · '
            'Jeffreys por anotação, média ± t·DP/√n por pessoa | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-01-dois-numeros.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ══ GRÁFICO 2 · quanto a amostra encolhe, e onde o artigo nos coloca ══
fig2, ax2 = plt.subplots(figsize=(11, 6.5))

def n_efetivo(e, rho):
    return e['registros'] / (1 + (e['por_pessoa'] - 1) * rho)

rhos = np.linspace(0, 1, 400)
los = np.array([ic(jeffreys(e2['p'], n_efetivo(e2, r)))[0] * 100 for r in rhos])
his = np.array([ic(jeffreys(e2['p'], n_efetivo(e2, r)))[1] * 100 for r in rhos])

ax2.fill_between(rhos, los, his, color=VERDE, alpha=0.20)
ax2.plot(rhos, los, color=VERDE, linewidth=2)
ax2.plot(rhos, his, color=VERDE, linewidth=2)
ax2.axhline(e2['p'] * 100, color='#333333', linewidth=2)
ax2.axhspan(e2['ic_pess'][0] * 100, e2['ic_pess'][1] * 100, color=VERMELHO, alpha=0.10)
ax2.axvline(icc_bruta, color=VERMELHO, linestyle='--', linewidth=1.8)
ax2.plot([icc_bruta, icc_bruta], [lo_ef * 100, hi_ef * 100], 'o', color=VERMELHO,
         markersize=8, zorder=3)
ax2.text(icc_bruta + 0.02, 49.0,
         f'ρ derivada do DP publicado = {vir(icc_bruta, 2)}\n'
         f'[{vir(lo_ef * 100)}%; {vir(hi_ef * 100)}%]',
         fontsize=10, color=VERMELHO)
ax2.text(0.52, 43.35, f'estimativa publicada: {e2["p"] * 100:.0f}%',
         fontsize=10, color='#333333')
ax2.text(0.70, 36.2, f'ρ = 1: cada pessoa vale uma moeda', fontsize=10, color=VERDE,
         ha='left', va='top')

ax2.set_xlim(0, 1)
ax2.set_ylim(34, 52)
ax2.set_xlabel('ρ · correlação intraclasse')
ax2.set_ylabel('Intervalo de 95% para a proporção (%)')
ax2.set_title('O 43% depende de quem é a amostra, e o artigo diz quem é\n'
              '4.335 anotações vindas de 209 pessoas valem cerca de '
              f'{mil(round(n_ef))} observações',
              fontsize=13, pad=15)

plt.figtext(0.5, 0.005, 'Modelo próprio · efeito de desenho DEFF = 1 + (m − 1)·ρ, '
            'm = 20,74, ρ derivada de DP = 0,19 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-02-a-amostra.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ══ GRÁFICO 3 · a distribuição das pessoas ══
fig3, ax3 = plt.subplots(figsize=(11, 6.5))

x = np.linspace(0.001, 0.999, 1000)
y = pessoas.pdf(x)
ax3.plot(x * 100, y, color=VERDE, linewidth=2.5, label='calibrada no DP publicado (19 pontos)')
faixa = (x >= d10) & (x <= d90)
ax3.fill_between(x[faixa] * 100, y[faixa], color=VERDE, alpha=0.22)
ax3.plot(x * 100, pessoas_rob.pdf(x), color=VERDE, linewidth=1.6, linestyle=':',
         label=f'robustez, ruído descontado ({vir(sd_real * 100)} pontos)')
ax3.axvline(e2['p'] * 100, color=VERMELHO, linewidth=2.5)
ax3.text(e2['p'] * 100 + 1.5, max(y) * 0.40,
         f'a média publicada\n{e2["p"] * 100:.0f}%', fontsize=11, color=VERMELHO)
ax3.annotate('', xy=(d10 * 100, max(y) * 0.12), xytext=(d90 * 100, max(y) * 0.12),
             arrowprops=dict(arrowstyle='<->', color=DOURADO, lw=2))
ax3.text((d10 + d90) / 2 * 100, max(y) * 0.055,
         f'os 80% do meio: de {vir(d10 * 100)}% a {vir(d90 * 100)}%',
         ha='center', fontsize=11, color=DOURADO, fontweight='bold')

ax3.set_xlim(0, 100)
ax3.set_ylim(bottom=0)
ax3.set_xlabel('Comportamentos habituais por pessoa (%)')
ax3.set_ylabel('Densidade entre pessoas')
ax3.set_title('A média é firme. O dia médio não existe.\n'
              f'Beta(α = {vir(alfa, 2)}; β = {vir(beta_, 2)}), calibrada na média 43% '
              'e no DP 19 publicados',
              fontsize=13, pad=15)
ax3.legend(frameon=False, fontsize=10, loc='upper right')

plt.figtext(0.5, 0.005, 'Modelo próprio · Beta pelo método dos momentos sobre '
            'Wood, Quinn & Kashy (2002), Tabela 2 | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-03-o-seu-dia.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# ══ GRÁFICO 4 · dois mundos ══
fig4, ax4 = plt.subplots(figsize=(11, 6.5))

ks = np.arange(MOMENTOS + 1)
freq_real = np.array([(hab == k).mean() for k in ks]) * 100
freq_med = np.array([(hab_med == k).mean() for k in ks]) * 100

larg = 0.40
ax4.bar(ks - larg / 2, freq_med, larg, color=CINZA, alpha=0.75,
        label='se todo mundo fosse a média')
ax4.bar(ks + larg / 2, freq_real, larg, color=VERDE, alpha=0.9,
        label='com as pessoas sendo diferentes')

ax4.set_xticks(ks)
ax4.set_xlabel(f'Momentos habituais, de {MOMENTOS} observados')
ax4.set_ylabel('Proporção das sequências (%)')
ax4.set_title('A mesma média, dois mundos diferentes\n'
              f'{mil(SORTEIOS)} sequências simuladas, semente {SEMENTE}',
              fontsize=13, pad=15)
ax4.legend(frameon=False, fontsize=10.5)

ax4.annotate(f'sequência sem nenhum habitual:\n{vir(freq_med[0], 2)}% contra '
             f'{vir(freq_real[0])}%',
             xy=(0 + larg / 2, freq_real[0] + 0.3), xytext=(-0.4, 16.0), ha='left',
             fontsize=10.5, color=VERMELHO,
             arrowprops=dict(arrowstyle='->', color=VERMELHO, lw=1.6))

plt.figtext(0.5, 0.005, 'Modelo próprio · Monte Carlo com semente declarada · '
            'momentos, não horas | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-111-grafico-04-dois-mundos.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**O número está certo. Só não responde à sua pergunta.**

Contando cada anotação como uma observação, o 43% vem com um intervalo de 41,5% a
44,5%. Contando pessoa, que é a unidade que o desenho do estudo realmente tem, ele vai
de 40,4% a 45,6%. Abre, mas pouco. A média é firme.

O mesmo artigo mediu **35% num estudo e 43% no outro**, com protocolos parecidos mas
não iguais. Contando pessoas, a diferença fica entre 2,8 e 13,2 pontos e não passa por
zero. Ela é real, e tem explicação publicada: no Estudo 2 a pessoa podia listar vários
comportamentos por sinal, e isso puxa mais hábitos para dentro. Não é sorteio. É
protocolo.

O que ninguém repete é o terceiro número do artigo: o desvio-padrão entre as pessoas,
**19 pontos**. Calibrando uma distribuição nele, **os 80% do meio vão de 18% a 69%**
dos comportamentos no automático. Não é uma faixa. É quase metade da escala.

Em dez momentos observados, a média é a mesma nos dois mundos: 4,3. Mas num mundo em
que todos fossem a média, quase ninguém sai de 2 a 6. Com as pessoas sendo o que são,
o intervalo abre para 1 a 8.

O 43% responde bem a "qual é a média desse grupo?" e mal a "quanto da minha vida é
hábito?". Uma média diz onde fica o centro do grupo. Não diz onde você está.

*Você é dos 18% ou dos 69%? E quando foi a última vez que você verificou?*

---

### ⚠️ Limitações do Modelo

- **A Beta da camada 3 é calibrada no DP publicado, que inclui ruído amostral.** Cada
  proporção por pessoa vem de cerca de 21 anotações, então parte dos 19 pontos é
  variação de amostragem dentro da pessoa, não diferença entre pessoas. Descontando
  esse ruído, o DP verdadeiro fica perto de 16 pontos, a ICC implícita cai de 0,15
  para 0,10 e os 80% do meio encolhem para cerca de 23% a 64%. A conclusão não muda;
  os números da camada 3 são o cenário mais largo.
- **Comportamentos não são horas.** O estudo classificou comportamentos relatados, e no
  Estudo 2 um mesmo sinal podia trazer mais de um. Por isso a camada 4 fala em momentos
  observados, e nenhuma linha deste notebook traduz 43% em horas do dia.
- **"Automático" é linguagem narrativa.** O critério operacional é frequência (quase
  todo dia) e contexto (mesmo lugar). Os autores estudaram a relação disso com
  pensamento e automaticidade, mas 43% habitual não significa 43% do dia sem
  consciência.
- **A amostra é de estudantes universitários**, num único campus, em 2002. Rotina de
  estudante tem estabilidade de contexto diferente da de quem tem filho pequeno, turno
  variável ou trabalho remoto.
- **Um e dois dias de observação.** Hábito é definido por repetição, e dois dias são
  pouco para observar repetição. O critério depende de o participante afirmar que faz
  aquilo quase todo dia, não de o estudo ter visto.
- **Classificação pelo próprio participante.** É a escolha certa para o construto, e é
  por isso que o ×0,80 não se aplica, mas significa que "habitual" aqui é o que a
  pessoa reconhece como habitual. O hábito que você não percebe não entra na conta,
  e ele empurra o número para cima.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*